# Gold 02 — Product performance

**Grain:** one row per order date, store, and product, for completed orders only.

Each order's discount is distributed across its item lines in proportion to pre-discount line value. The final line receives any rounding remainder, so summed product revenue exactly equals completed order revenue. Estimated gross profit is that allocated net line revenue minus Silver estimated line cost. Tax is excluded.


In [ ]:
import os  # Set Java before Spark starts.
import shutil  # Find Linux Java.
from pathlib import Path  # Build data paths.
from pyspark.sql import SparkSession, functions as F  # Use Spark and column expressions.
from delta import configure_spark_with_delta_pip  # Add Delta Lake to Spark.
java = shutil.which("java")  # Locate WSL Java.
if java is None:  # Require Java.
    raise RuntimeError("Java is unavailable in WSL")  # Explain a startup failure.
os.environ["JAVA_HOME"] = str(Path(java).resolve().parent.parent)  # Avoid the Windows Java path.
project_root = Path("/mnt/f/Big Data/Retail_Lakehouse_Learning")  # Find the project in WSL.
if not project_root.is_dir():  # Check the mounted drive.
    raise FileNotFoundError(project_root)  # Stop on an incorrect path.
builder = (SparkSession.builder.master("local[2]")  # Use two local threads.
    .appName("RetailGoldDailyStoreSales")  # Label this job.
    .config("spark.sql.shuffle.partitions", "4")  # Keep small local shuffles manageable.
    .config("spark.ui.enabled", "false")  # Avoid a second Spark UI port.
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")  # Enable Delta SQL.
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))  # Register Delta.
spark = configure_spark_with_delta_pip(builder).getOrCreate()  # Start or reuse Spark.
print("Spark:", spark.version, "Timezone:", spark.conf.get("spark.sql.session.timeZone"))  # Record session behavior.


In [ ]:
silver_root = project_root / "lakehouse" / "silver" / "data"  # Locate cleaned source tables.
gold_root = project_root / "lakehouse" / "gold" / "data"  # Locate the daily control totals.
orders_df = spark.read.format("delta").load(str(silver_root / "orders"))  # Read one row per order.
items_df = spark.read.format("delta").load(str(silver_root / "order_items"))  # Read one row per item.
products_df = spark.read.format("delta").load(str(silver_root / "products"))  # Read product names and categories.
daily_df = spark.read.format("delta").load(str(gold_root / "daily_store_sales"))  # Read previously checked daily KPIs.
if "estimated_line_cost" not in items_df.columns:  # Require completed Silver cost work.
    raise ValueError("Rerun Silver order_items before Gold")  # Prevent a false profit result.
if products_df.count() != products_df.select("product_id").distinct().count():  # Require a unique dimension key.
    raise ValueError("Duplicate Silver product IDs")  # Avoid duplicated item revenue.
completed_df = (orders_df.filter(F.col("status") == "completed")  # Keep completed orders.
    .select("order_id", "store_id", F.to_date("order_ts").alias("date"), "subtotal", "discount_amount"))  # Keep allocation fields.
if completed_df.filter(F.col("subtotal").isNull() | (F.col("subtotal") <= 0) | F.col("discount_amount").isNull()).count():  # Need a valid denominator.
    raise ValueError("Completed orders have invalid allocation amounts")  # Stop unsafe division.
print("Completed orders:", completed_df.count(), "Silver items:", items_df.count())  # Show input sizes.


In [ ]:
from pyspark.sql import Window  # Define each order's line allocation window.
lines_df = items_df.join(completed_df, on="order_id", how="inner")  # Attach completed order fields to item rows.
if lines_df.count() != items_df.join(completed_df.select("order_id"), "order_id", "inner").count():  # Confirm expected join size.
    raise ValueError("Unexpected completed-item join count")  # Surface changed source keys.
if lines_df.filter(F.col("line_total").isNull() | F.col("estimated_line_cost").isNull()).count():  # Require inputs for revenue and cost.
    raise ValueError("Completed items lack price or cost")  # Keep Gold complete.
order_window = Window.partitionBy("order_id")  # Group all lines from one order.
last_line_window = Window.partitionBy("order_id").orderBy(F.col("order_item_id").desc())  # Pick one line for the cents remainder.
allocated_df = (lines_df  # Start from completed item lines.
    .withColumn("rough_discount", F.bround(F.col("discount_amount") * F.col("line_total") / F.col("subtotal"), 2).cast("decimal(18,2)"))  # Round each proportional discount.
    .withColumn("rough_discount_sum", F.sum("rough_discount").over(order_window))  # Find the rounded total within each order.
    .withColumn("last_line", F.row_number().over(last_line_window))  # Mark one line to receive the remainder.
    .withColumn("allocated_discount", (F.col("rough_discount") + F.when(F.col("last_line") == 1, F.col("discount_amount") - F.col("rough_discount_sum")).otherwise(F.lit(0))).cast("decimal(18,2)"))  # Correct cents.
    .withColumn("line_revenue", (F.col("line_total") - F.col("allocated_discount")).cast("decimal(18,2)"))  # Net sales before tax.
    .withColumn("estimated_line_gross_profit", (F.col("line_revenue") - F.col("estimated_line_cost")).cast("decimal(18,2)")))  # Estimate profit.
allocation_check_df = allocated_df.groupBy("order_id").agg(  # Recombine lines to validate allocation.
    F.sum("line_total").cast("decimal(18,2)").alias("line_subtotal"),  # Compare with order subtotal.
    F.sum("allocated_discount").cast("decimal(18,2)").alias("line_discount"))  # Compare with order discount.
bad_allocations = (completed_df.join(allocation_check_df, "order_id", "left")  # Preserve every completed order.
    .filter(F.col("line_subtotal").isNull() | (F.abs(F.col("line_subtotal") - F.col("subtotal")) > F.lit(0.02)) | (F.col("line_discount") != F.col("discount_amount"))).count())  # Count failures.
if bad_allocations:  # Enforce exact discount reconciliation.
    raise ValueError(f"{bad_allocations} completed orders failed discount allocation")  # Prevent wrong product KPIs.
print("Completed item lines:", allocated_df.count(), "Bad allocations:", bad_allocations)  # Show the check.


In [ ]:
product_lines_df = allocated_df.join(products_df.select("product_id", "product_name", "category"), "product_id", "left")  # Add product labels.
if product_lines_df.filter(F.col("product_name").isNull()).count():  # Check every product ID exists.
    raise ValueError("Completed item refers to unknown product")  # Keep labels and totals complete.
product_gold_df = product_lines_df.groupBy("date", "store_id", "product_id", "product_name", "category").agg(  # One row per date, store, product.
    F.countDistinct("order_id").alias("orders"),  # Count orders that contain this product.
    F.sum("quantity").alias("units_sold"),  # Count product units.
    F.sum("line_revenue").cast("decimal(18,2)").alias("line_revenue"),  # Sum allocated net sales.
    F.sum("estimated_line_cost").cast("decimal(18,2)").alias("estimated_cost"),  # Sum estimated line costs.
    F.sum("estimated_line_gross_profit").cast("decimal(18,2)").alias("estimated_gross_profit"))  # Sum line profit.
if product_gold_df.count() != product_gold_df.select("date", "store_id", "product_id").distinct().count():  # Check Gold grain.
    raise ValueError("Duplicate date/store/product Gold keys")  # Stop before writing.
daily_revenue = daily_df.agg(F.sum("revenue").alias("v")).first().v  # Use trusted daily net sales.
product_revenue = product_gold_df.agg(F.sum("line_revenue").alias("v")).first().v  # Sum product net sales.
daily_profit = daily_df.agg(F.sum("estimated_gross_profit").alias("v")).first().v  # Use daily profit control.
product_profit = product_gold_df.agg(F.sum("estimated_gross_profit").alias("v")).first().v  # Sum product profit.
if product_revenue != daily_revenue or product_profit != daily_profit:  # Require the same money in both marts.
    raise ValueError(f"Product totals do not match daily: revenue {product_revenue}/{daily_revenue}, profit {product_profit}/{daily_profit}")  # Explain differences.
print("Product rows:", product_gold_df.count(), "Revenue:", product_revenue, "Profit:", product_profit)  # Show reconciliation.
product_gold_df.orderBy("date", "store_id", "product_id").show(5, truncate=False)  # Inspect sample rows.


In [ ]:
gold_path = gold_root / "product_performance"  # Choose the product Gold folder.
gold_path.parent.mkdir(parents=True, exist_ok=True)  # Ensure the ignored data folder exists.
product_gold_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(str(gold_path))  # Commit one Gold snapshot.
saved_df = spark.read.format("delta").load(str(gold_path))  # Reopen saved Delta data.
if saved_df.count() != product_gold_df.count():  # Check persisted row count.
    raise ValueError("Saved product Gold row count differs")  # Surface write failure.
if saved_df.agg(F.sum("line_revenue").alias("v")).first().v != daily_revenue:  # Check saved financial total.
    raise ValueError("Saved product revenue differs from daily")  # Surface incorrect data.
print("Saved product_performance:", saved_df.count(), "Delta log:", (gold_path / "_delta_log").is_dir())  # Confirm Delta commit.
saved_df.printSchema()  # Show analytical types.
